# DTLS image super-resolution on Google Colab

This notebook downloads the DTLS source code, the generalized natural-low-resolution checkpoint, and the example images supplied by the authors. It then runs inference on a Colab GPU and displays the generated images.

**Before running:** in Colab select **Runtime → Change runtime type → T4 GPU** (or another GPU).

## 1. Install dependencies and clone the repository

In [ ]:
!git clone -q https://github.com/GreyCC/DTLS_1024.git /content/DTLS_1024
%cd /content/DTLS_1024
!python -m pip install -q einops imgaug gdown lmdb opencv-python tqdm wandb

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Select a GPU runtime in Runtime → Change runtime type.")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Download the generalized checkpoint and example images

The checkpoint is the generalized model for natural low-resolution face super-resolution. The example images are downloaded from the shared Google Drive folder. Google Drive permissions must allow viewers to download the files.

In [ ]:
CHECKPOINT_ID = "1VxzoGEFfH96L9YHUmaxmlBpRJUUUugzE"
EXAMPLE_FOLDER_ID = "1bDMjDNe_bUEnQIeqS3alBOvtCRLvpZhv"

weights_dir = Path("/content/DTLS_1024/pretrained_weights")
weights_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = weights_dir / "DTLS_generalized_NLR.pth"

if not checkpoint_path.exists():
    subprocess.run(["gdown", "--id", CHECKPOINT_ID, "-O", str(checkpoint_path)], check=True)

download_dir = Path("/content/DTLS_1024/downloaded_examples")
if download_dir.exists():
    shutil.rmtree(download_dir)
download_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(["gdown", "--folder", EXAMPLE_FOLDER_ID, "-O", str(download_dir)], check=True)

input_dir = Path("/content/DTLS_1024/NLQ_Faces")
if input_dir.exists():
    shutil.rmtree(input_dir)
input_dir.mkdir(parents=True, exist_ok=True)

extensions = {".jpg", ".jpeg", ".png"}
images = [p for p in download_dir.rglob("*") if p.is_file() and p.suffix.lower() in extensions]
if not images:
    raise FileNotFoundError("No JPG, JPEG, or PNG files were found in the Google Drive folder.")

from PIL import Image

for image in images:
    # Normalize RGBA/grayscale files to RGB because the DTLS encoder expects 3 channels.
    with Image.open(image) as opened_image:
        opened_image.convert("RGB").save(input_dir / image.name)

print(f"Downloaded {len(images)} input image(s).")
print("Checkpoint:", checkpoint_path, f"({checkpoint_path.stat().st_size / 1024**2:.1f} MB)")

## 3. Run super-resolution

The repository evaluator saves results below `eval/DTLS_colab_generalized/`. The evaluator currently outputs 512×512 images.

In [ ]:
output_name = "DTLS_colab_generalized"
command = [
    "python", "eval.py",
    "--path", str(input_dir),
    "--cuda", "0",
    "--batch_size", "1",
    "--workers", "0",
    "--output_path", output_name,
    "--ckpt", str(checkpoint_path),
]
print("Running:", " ".join(command))
completed = subprocess.run(command, cwd="/content/DTLS_1024", text=True, capture_output=True)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(f"eval.py failed with exit code {completed.returncode}. See the full stderr above.")
print(f"Results saved to /content/DTLS_1024/eval/{output_name}")

## 4. Display the results

In [ ]:
from IPython.display import display
from PIL import Image

result_dir = Path("/content/DTLS_1024/eval") / output_name
result_paths = sorted(p for p in result_dir.iterdir() if p.suffix.lower() in extensions)

for result_path in result_paths:
    print(result_path.name)
    display(Image.open(result_path))

## Use your own images

Replace the download cell with the following cell, upload images through the Colab file picker, and then rerun the inference and display cells:

```python
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    shutil.copy2(name, input_dir / Path(name).name)
```

The experimental-study checkpoint can also be tested by changing `CHECKPOINT_ID` to `1Uy9YxYiLJ-c5rqWmOtHCivn5LnWK5rZK`.